# Export Randomly Flipped ONNX from Top-Weight Candidates

This notebook samples **k random bit flips** from a top-weight candidate list and exports flipped ONNX model(s).

It reuses existing code from:
- `op_0103/export_flipped_onnx.py`
- `op_0103/important_bits_onnx.py`

Supported controls:
- input top-weight JSON file
- choose top-N weights from that file
- choose target model: `vision`, `policy`, or `both`
- choose number of random flips `k`
- choose bit range with the same syntax as the main code, e.g. `sign`, `exponent_sign`, `all`, `>=8`, `0,1,15`
- deterministic random seed

In [1]:
from pathlib import Path
import json
import os
import random
import sys

import pandas as pd

REPO_ROOT = Path('/home/zx/Projects/DAC/STAFI')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from op_0103.export_flipped_onnx import (
    DEFAULT_OUT_DIR,
    DEFAULT_POLICY_ONNX,
    DEFAULT_VISION_ONNX,
    FlipSpec,
    export_one,
)
from op_0103.important_bits_onnx import parse_bitset_with_mode


In [2]:
# Config
weights_json = "/home/zx/Projects/DAC/STAFI/op_0103/weights_3000_with_bias.json"
target_model = "vision"      # "vision", "policy", or "both"
top_n_weights = 50
k_flips = 2
bit_spec = "sign"            # examples: "sign", "exponent_sign", ">=8", "all", "0,1,15"
allow_repeat_weight = False   # False means one selected weight can only contribute one flipped bit
strict = False

# Seed control (manual)
seed_mode = "random"          # set to "fixed" or "random"
seed_value = 0                # used only when seed_mode == "fixed"

vision_onnx = Path(DEFAULT_VISION_ONNX)
policy_onnx = Path(DEFAULT_POLICY_ONNX)
out_dir = Path(DEFAULT_OUT_DIR) / "flipped_models"
out_prefix = "random_topweights"


In [3]:
def load_weight_candidates(path: str):
    path = Path(path)
    payload = json.loads(path.read_text())
    rows = payload.get("candidates")
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"No valid 'candidates' in {path}")
    return payload, rows


def filter_top_weights(rows, target_model: str, top_n: int):
    if target_model not in {"vision", "policy", "both"}:
        raise ValueError(f"Unsupported target_model: {target_model}")
    if top_n <= 0:
        raise ValueError("top_n_weights must be >= 1")

    if target_model == "both":
        filtered = [r for r in rows if r.get("model") in {"vision", "policy"}]
    else:
        filtered = [r for r in rows if r.get("model") == target_model]

    if not filtered:
        raise ValueError(f"No candidates found for target_model={target_model}")
    return filtered[: min(top_n, len(filtered))]


def sample_flip_specs(weight_rows, bit_spec: str, k: int, seed: int, allow_repeat_weight: bool):
    bits, bitset_mode = parse_bitset_with_mode(bit_spec)
    if not bits:
        raise ValueError(f"No bits resolved from bit_spec={bit_spec!r}")
    if k <= 0:
        raise ValueError("k_flips must be >= 1")

    rng = random.Random(seed)
    sampled = []

    if allow_repeat_weight:
        for _ in range(k):
            row = rng.choice(weight_rows)
            bit = rng.choice(bits)
            sampled.append(
                FlipSpec(
                    model=str(row["model"]),
                    name=str(row["name"]),
                    flat=int(row["flat"]),
                    bit=int(bit),
                )
            )
    else:
        if k > len(weight_rows):
            raise ValueError(
                f"k_flips={k} exceeds available unique weights={len(weight_rows)} when allow_repeat_weight=False"
            )
        chosen_rows = rng.sample(weight_rows, k)
        for row in chosen_rows:
            bit = rng.choice(bits)
            sampled.append(
                FlipSpec(
                    model=str(row["model"]),
                    name=str(row["name"]),
                    flat=int(row["flat"]),
                    bit=int(bit),
                )
            )

    return sampled, bits, bitset_mode


def group_specs_by_model(specs):
    grouped = {}
    for spec in specs:
        grouped.setdefault(spec.model, []).append(spec)
    return grouped


def build_output_path(out_dir: Path, model_key: str, top_n_weights: int, k_flips: int, bitset_mode: str, seed: int):
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = f"{model_key}_model_{out_prefix}_top{top_n_weights}_k{k_flips}_{bitset_mode}_seed{seed}.onnx"
    return out_dir / fname

def resolve_seed(seed_mode: str, seed_value: int) -> int:
    if seed_mode == "fixed":
        return int(seed_value)
    if seed_mode == "random":
        return int(random.SystemRandom().randrange(0, 2**31 - 1))
    raise ValueError(f"Unsupported seed_mode: {seed_mode}")


In [4]:
# Load top-weight file and sample random flips
payload, candidate_rows = load_weight_candidates(weights_json)
selected_weights = filter_top_weights(candidate_rows, target_model=target_model, top_n=top_n_weights)

resolved_seed = resolve_seed(seed_mode, seed_value)

sampled_specs, bits, bitset_mode = sample_flip_specs(
    selected_weights,
    bit_spec=bit_spec,
    k=k_flips,
    seed=resolved_seed,
    allow_repeat_weight=allow_repeat_weight,
)
by_model = group_specs_by_model(sampled_specs)

print("weights_json =", weights_json)
print("target_model =", target_model)
print("top_n_weights =", top_n_weights)
print("k_flips =", k_flips)
print("bit_spec =", bit_spec)
print("resolved_bits =", bits)
print("bitset_mode =", bitset_mode)
print("seed_mode =", seed_mode)
print("seed =", resolved_seed)
print("allow_repeat_weight =", allow_repeat_weight)
print("selected candidate count =", len(selected_weights))
print("sampled flips by model =", {k: len(v) for k, v in by_model.items()})


weights_json = /home/zx/Projects/DAC/STAFI/op_0103/weights_3000_with_bias.json
target_model = vision
top_n_weights = 50
k_flips = 2
bit_spec = sign
resolved_bits = [15]
bitset_mode = sign
seed_mode = random
seed = 683958345
allow_repeat_weight = False
selected candidate count = 50
sampled flips by model = {'vision': 2}


In [5]:
# Preview sampled flips
preview_rows = []
for idx, spec in enumerate(sampled_specs, start=1):
    preview_rows.append({
        "idx": idx,
        "model": spec.model,
        "name": spec.name,
        "flat": spec.flat,
        "bit": spec.bit,
    })

preview_df = pd.DataFrame(preview_rows)
preview_df

,idx,model,name,flat,bit
0,1,vision,vision_model.vision._en.stages.1.downsample.pr...,4681,15
1,2,vision,vision_model.vision._en.stages.1.downsample.pr...,15305,15


In [6]:
# Export ONNX model(s)
inputs = {
    "vision": os.path.abspath(str(vision_onnx)),
    "policy": os.path.abspath(str(policy_onnx)),
}

exported = {}
for model_key, model_specs in by_model.items():
    onnx_out = build_output_path(
        out_dir=out_dir,
        model_key=model_key,
        top_n_weights=top_n_weights,
        k_flips=len(model_specs),
        bitset_mode=bitset_mode,
        seed=resolved_seed,
    )
    export_one(
        model_key=model_key,
        onnx_in=inputs[model_key],
        onnx_out=str(onnx_out),
        specs=model_specs,
        strict=strict,
    )
    exported[model_key] = str(onnx_out)

exported

[Done] target_model=vision
[Done] input=/home/zx/Projects/DAC/STAFI/op_0103/models/driving_vision.onnx
[Done] output=/home/zx/Projects/DAC/STAFI/op_0103/out/flipped_models/vision_model_random_topweights_top50_k2_sign_seed683958345.onnx
[Done] requested=2 applied=2 skipped=0 non_finite_new=0
  idx=0 name=vision_model.vision._en.stages.1.downsample.proj.1.reparam_conv.weight flat=4681 bit=15 old=0.0013980865 new=-0.0013980865 finite=True
  idx=1 name=vision_model.vision._en.stages.1.downsample.proj.1.reparam_conv.weight flat=15305 bit=15 old=-0.0010318756 new=0.0010318756 finite=True


{'vision': '/home/zx/Projects/DAC/STAFI/op_0103/out/flipped_models/vision_model_random_topweights_top50_k2_sign_seed683958345.onnx'}

In [7]:
# Optional: save the sampled random flip plan for reproducibility
sample_plan = {
    "meta": {
        "weights_json": os.path.abspath(weights_json),
        "target_model": target_model,
        "top_n_weights": int(top_n_weights),
        "k_flips": int(k_flips),
        "bit_spec": bit_spec,
        "bitset_mode": bitset_mode,
        "resolved_bits": [int(x) for x in bits],
        "seed_mode": seed_mode,
        "seed": int(resolved_seed),
        "allow_repeat_weight": bool(allow_repeat_weight),
    },
    "sampled_flips": preview_rows,
    "exported": exported,
}

plan_json_out = out_dir / f"sample_plan_{out_prefix}_top{top_n_weights}_k{k_flips}_{bitset_mode}_seed{resolved_seed}.json"
plan_json_out.write_text(json.dumps(sample_plan, indent=2), encoding="utf-8")
print(plan_json_out)


/home/zx/Projects/DAC/STAFI/op_0103/out/flipped_models/sample_plan_random_topweights_top50_k2_sign_seed683958345.json
